In [ ]:
# ==========================================
# 1. COLAB SETUP: DRIVE + REPO CODE
# ==========================================
import os
import pathlib
import subprocess
import sys

from google.colab import drive
from google.colab import userdata

REPO_URL = "https://github.com/DrHBSB/RadLE_CRASH_Lab.git"
REPO_REF = "codex/radle-argon4-colab-20261016"
REPO_DIR = pathlib.Path("/content/RadLE_CRASH_Lab")
SRC_DIR = REPO_DIR / "src"
MODULE_PATH = SRC_DIR / "radle_benchmark.py"
github_token = None

# Drive remains the home for images and benchmark outputs.
drive.mount("/content/drive")


def run_git(args, check=True):
    """Run git and print useful stderr without leaking the optional token."""
    result = subprocess.run(args, text=True, capture_output=True)
    stdout = result.stdout.replace(github_token, "***") if github_token else result.stdout
    stderr = result.stderr.replace(github_token, "***") if github_token else result.stderr
    if stdout.strip():
        print(stdout)
    if stderr.strip():
        print(stderr)
    if check and result.returncode != 0:
        hint = ""
        if "403" in stderr or "Write access to repository not granted" in stderr:
            hint = (
                " The GITHUB_TOKEN secret exists, but GitHub rejected it. "
                "Create a token that has read access to this private repository "
                "and permission to read repository contents."
            )
        raise RuntimeError(f"Git command failed with exit code {result.returncode}: {args[:2]}.{hint}")
    return result


# Fetch the immutable release tag and load exactly its code.
if not (REPO_DIR / ".git").exists():
    run_git(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)])
run_git(["git", "-C", str(REPO_DIR), "fetch", "origin", "tag", REPO_REF])
run_git(["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_REF])
if run_git(["git", "-C", str(REPO_DIR), "status", "--porcelain"]).stdout.strip():
    raise RuntimeError("Code checkout has local edits; use a fresh Colab runtime before collection.")

if not MODULE_PATH.exists():
    raise RuntimeError(f"Benchmark module not found after setup: {MODULE_PATH}")

current_ref = run_git(["git", "-C", str(REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD"]).stdout.strip()
current_commit = run_git(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"]).stdout.strip()
print("Repo checkout:", current_ref, current_commit)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# radle_benchmark imports all provider adapters, while this run sends only through OpenRouter.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "anthropic", "google-genai"], check=True)


In [ ]:
# ==========================================
# 2. API CLIENT + BENCHMARK IMPORTS
# ==========================================
import importlib
import inspect

from openai import OpenAI

# Reload dependencies too: git pull does not refresh cached Python modules.
import radle_reasoning_ceiling
import radle_retry_policy
import radle_job_queue
import radle_attempt_artifacts
import radle_attempt_ledger
import radle_error_evidence
import radle_journal_recovery
import radle_concurrent_collection
import radle_benchmark

importlib.invalidate_caches()
radle_attempt_artifacts = importlib.reload(radle_attempt_artifacts)
radle_attempt_ledger = importlib.reload(radle_attempt_ledger)
radle_error_evidence = importlib.reload(radle_error_evidence)
radle_reasoning_ceiling = importlib.reload(radle_reasoning_ceiling)
radle_retry_policy = importlib.reload(radle_retry_policy)
radle_job_queue = importlib.reload(radle_job_queue)
radle_journal_recovery = importlib.reload(radle_journal_recovery)
radle_concurrent_collection = importlib.reload(radle_concurrent_collection)

radle_benchmark = importlib.reload(radle_benchmark)
create_scorer_view = radle_benchmark.create_scorer_view
run_benchmark = radle_benchmark.run_benchmark
audit_benchmark_output = radle_benchmark.audit_benchmark_output
run_targeted_repair = radle_benchmark.run_targeted_repair
run_autonomous_openrouter_workflow = radle_benchmark.run_autonomous_openrouter_workflow

for _required_param in ("anthropic_client", "gemini_client", "backup_dir"):
    if _required_param not in inspect.signature(run_benchmark).parameters:
        raise RuntimeError(
            f"Loaded stale radle_benchmark missing {_required_param}. "
            "Rerun the setup/import cells after git pull, or restart the runtime."
        )

for _required_function in (
    "audit_benchmark_output",
    "run_targeted_repair",
    "build_run_paths",
    "promote_final_results",
    "export_public_release_tables",
    "run_autonomous_openrouter_workflow",
    "assert_openrouter_run_gate",
    "choose_repair_input_csv",
):
    if not hasattr(radle_benchmark, _required_function):
        raise RuntimeError(
            f"Loaded stale radle_benchmark missing {_required_function}. "
            "Rerun the setup/import cells after git pull, or restart the runtime."
        )

print(f"Loaded benchmark module: {radle_benchmark.__file__}")

_openrouter_key = userdata.get("OPENROUTER_API_KEY")
if not _openrouter_key:
    raise RuntimeError("Add the fresh OPENROUTER_API_KEY in Colab Secrets and grant notebook access.")
openrouter_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=_openrouter_key,
                           default_headers={"X-OpenRouter-Cache": "false"}, max_retries=0)
openai_client = anthropic_client = gemini_client = None
print("OpenRouter client ready using OPENROUTER_API_KEY")


In [ ]:
# 3. GEMINI 4 ARGON: SAME CASES, SEPARATE RESUMABLE RUN
import copy, sys, importlib
from pathlib import Path
sys.path.insert(0, str(REPO_DIR / "scripts"))
import smoke_openrouter_16_precolab as metadata_client
import radle_argon4
radle_argon4 = importlib.reload(radle_argon4)

EXPECTED_CASES = 200
RUN_LABEL = radle_argon4.RUN_LABEL
CONCURRENCY = 1
ACTIVE_MODELS = copy.deepcopy(radle_argon4.MODELS)
RETRY_AUTHORIZATION = copy.deepcopy(radle_argon4.RETRY_AUTHORIZATION)
dataset_root = Path("/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset")
# Metadata only; billing is checked again by the collector before paid requests.
live_route = radle_argon4.preflight(_openrouter_key, metadata_client.request_json)
run_paths = radle_argon4.prepare(radle_benchmark, dataset_root, live_route)
print("READY: Gemini 4 Argon, cases 1–200, output cap 32768, concurrency 1")
print("Reasoning: high -> medium -> low; step down only after two confirmed ceilings")
print("Maximum: 2 ordinary + 2 spaced repairs = at most 4 paid attempts per pair")
print("Held: refusals, unknown remote outcomes, billing and local blocks")
print("Output:", run_paths["run_root"])
print("Resume this RUN_LABEL and policy ID; never change either to repeat retries.")
print("Runtime release:", REPO_REF)


In [ ]:
# 4. COLLECT, THEN RUN THE SINGLE DURABLE TWO-RETRY PASS
from IPython.display import display

workflow_options = dict(
    client=openrouter_client, dataset_root=dataset_root, run_paths=run_paths,
    models=ACTIVE_MODELS, expected_cases=EXPECTED_CASES, run_label=RUN_LABEL,
    expected_returned_model_ids={m["name"]: set(m["expected_returned_model_ids"]) for m in ACTIVE_MODELS},
    expected_providers={m["name"]: m["expected_provider"] for m in ACTIVE_MODELS},
    expected_reasoning_rows={m["name"]: "all" for m in ACTIVE_MODELS},
    smoke_first=False, full_run=True, concurrency=CONCURRENCY,
    max_job_attempts=2, max_output_tokens=32768, max_repair_passes=0,
    migration=None, resume_blocked=True, repair_once=False, audit_only=False,
    allow_missing_reasoning=False, reuse_smoke_results=False,
    promote_private=False, export_public=False,
)
# First preserve/use ordinary authority. Only afterward grant the same durable repair ID.
# A repeated Run all cannot create another repair grant or resend saved answers.
AUTO_WORKFLOW_RESULT = run_autonomous_openrouter_workflow(**workflow_options)
if AUTO_WORKFLOW_RESULT.get("unresolved_pairs"):
    print("\n=== AUTHORIZED SPACED REPAIRS: UP TO TWO PER ELIGIBLE FAILED PAIR ===")
    AUTO_WORKFLOW_RESULT = run_autonomous_openrouter_workflow(
        **workflow_options, retry_authorization=RETRY_AUTHORIZATION)

df_final = AUTO_WORKFLOW_RESULT["final_df"]
print("\nWORKFLOW STATUS:", AUTO_WORKFLOW_RESULT.get("status", "complete"))
print("Result source:", AUTO_WORKFLOW_RESULT["final_source_csv"])
print("Rows:", len(df_final))
print("Unresolved pairs:", len(AUTO_WORKFLOW_RESULT.get("unresolved_pairs", [])))
display(AUTO_WORKFLOW_RESULT.get("failure_inventory"))
for section in ("dataset_integrity", "bucket_summary", "status_summary"):
    print(section)
    display(AUTO_WORKFLOW_RESULT["final_audit"][section])
print("Private promotion and public export are disabled. Preserve raw results, backups and journal.")
